In [ ]:
import pandas as pd
import numpy as np
import random
import folium
import time
import requests
import json
import os
from IPython.display import display, HTML

# 1. Konfigurasi File
files_matriks_jarak = {
    'Selatan': '../data/matriks_jarak_riil_selatan.csv',
    'Timur': '../data/matriks_jarak_riil_timur.csv',
    'Utara': '../data/matriks_jarak_riil_utara.csv',
    'Barat': '../data/matriks_jarak_riil_barat.csv',
    'Pusat': '../data/matriks_jarak_riil_pusat.csv'
}

files_matriks_waktu = {
    'Selatan': '../data/datamatriks_waktu_selatan.csv',
    'Timur': '../data/datamatriks_waktu_timur.csv',
    'Utara': '../data/datamatriks_waktu_utara.csv',
    'Barat': '../data/datamatriks_waktu_barat.csv',
    'Pusat': '../data/datamatriks_waktu_pusat.csv'
}

# Baca data koordinat
coords_df = pd.read_csv('../data/koordinat_eas.csv')

# Fungsi membersihkan koordinat rusak dari Excel
def fix_coord(val, is_lat=True):
    val = str(val).strip().replace('.', '').replace(',', '')
    if not val or val.lower() == 'nan': return 0.0
    try:
        num = float(val)
        if is_lat:
            if num > 0: num = -num
            while num <= -10: num /= 10.0
            if num == 0: num = -7.255
        else:
            while num >= 1000: num /= 10.0
            while 0 < num < 112: num *= 10.0
        return num
    except: return 0.0

coords_df['Latitude'] = coords_df['Latitude'].apply(lambda x: fix_coord(x, True))
coords_df['Longitude'] = coords_df['Longitude'].apply(lambda x: fix_coord(x, False))

if not coords_df['Nama Puskesmas'].str.contains('Gudang Farmasi', case=False, na=False).any():
    new_row = pd.DataFrame([{
        'Nama Puskesmas': 'UPTD Gudang Farmasi Surabaya', 
        'Latitude': -7.255, 
        'Longitude': 112.750, 
        'Wilayah': 'Pusat'
    }])
    coords_df = pd.concat([coords_df, new_row], ignore_index=True)

# Fungsi hitung prioritas (Jaringan Pelayanan & Jenis Layanan)
def build_priority_scores(places_list, df_coords):
    gamma_array = np.ones(len(places_list))
    for i, name in enumerate(places_list):
        if 'gudang farmasi' in name.lower():
            gamma_array[i] = 1.0
            continue
        match = df_coords[df_coords['Nama Puskesmas'].str.strip().str.lower() == name.strip().lower()]
        if not match.empty:
            jaringan = str(match.iloc[0].get('Jaringan Pelayanan', '')).strip().lower()
            layanan = str(match.iloc[0].get('Jenis Layanan', '')).strip().lower()
            if jaringan == 'puskesmas' and layanan == 'rawat inap': gamma_array[i] = 3.0
            elif jaringan == 'puskesmas' and layanan == 'rawat jalan': gamma_array[i] = 2.0
            elif jaringan == 'pustu' and layanan == 'rawat jalan': gamma_array[i] = 1.0
    return gamma_array

def singkat_nama(nama):
    nama = nama.strip()
    if nama.lower().startswith('puskesmas '): return 'P. ' + nama[10:]
    elif nama.lower().startswith('pustu '): return 'P. ' + nama[6:]
    return nama

def get_route_geometry(start_coords, end_coords):
    url = f"http://router.project-osrm.org/route/v1/driving/{start_coords[1]},{start_coords[0]};{end_coords[1]},{end_coords[0]}?overview=full&geometries=geojson"
    try:
        r = requests.get(url, timeout=5)
        res = r.json()
        if res['code'] == 'Ok':
            return [[c[1], c[0]] for c in res['routes'][0]['geometry']['coordinates']]
    except: pass
    return [start_coords, end_coords]

In [ ]:
def solve_aco(distance_matrix, time_matrix, priority_scores, num_ants=50, max_iter=150, alpha=1.0, beta=2.0, rho=0.1, Q=100):
    n = len(time_matrix)
    pheromone = np.ones((n, n)) * 0.1
    visibility = np.zeros((n, n))

    for i in range(n):
        for j in range(n):
            if i != j and time_matrix[i][j] > 0:
                visibility[i][j] = 1.0 / time_matrix[i][j]

    gamma_array = np.array(priority_scores, dtype=float)
    
    best_route = None
    best_time_total = float('inf')
    best_dist_total = float('inf')
    convergence_history = [] 

    for it in range(max_iter):
        routes, route_times = [], []
        current_best_iter_time = float('inf') 

        for ant in range(num_ants):
            route, visited, waktu_kurir = [0], {0}, 0 

            while len(visited) < n:
                curr = route[-1]
                probs = np.zeros(n)

                for j in range(n):
                    if j not in visited:
                        waktu_perjalanan = time_matrix[curr][j]
                        waktu_kembali = time_matrix[j][0]
                        
                        if waktu_kurir + waktu_perjalanan + 20 + waktu_kembali <= 600:
                            probs[j] = (pheromone[curr][j] ** alpha) * (visibility[curr][j] ** beta) * gamma_array[j]

                if probs.sum() == 0:
                    unvisited = list(set(range(n)) - visited)
                    if not unvisited: break
                    if curr == 0:
                        next_node = random.choice(unvisited)
                        waktu_kurir += time_matrix[curr][next_node] + 20
                        route.append(next_node); visited.add(next_node)
                    else:
                        route.append(0); waktu_kurir = 0 
                else:
                    next_node = np.random.choice(range(n), p=probs/probs.sum())
                    waktu_kurir += time_matrix[curr][next_node] + 20
                    route.append(next_node); visited.add(next_node)

            if route[-1] != 0: route.append(0)

            # --- PERHITUNGAN WAKTU EVALUASI YANG 100% AKURAT (+20 Menit) ---
            total_time = 0
            total_dist = 0
            for i in range(len(route) - 1):
                u, v = route[i], route[i+1]
                total_time += time_matrix[u][v]
                total_dist += distance_matrix[u][v]
                if v != 0: total_time += 20 # Waktu Pelayanan

            routes.append(route); route_times.append(total_time)

            if total_time < best_time_total:
                best_time_total = total_time; best_dist_total = total_dist; best_route = route 

        convergence_history.append(best_time_total)
        pheromone *= (1 - rho)
        
        for i in range(num_ants):
            d_tau = Q / route_times[i] 
            r = routes[i]
            for j in range(len(r) - 1):
                pheromone[r[j]][r[j+1]] += d_tau
                pheromone[r[j+1]][r[j]] += d_tau 

    return best_route, best_dist_total, best_time_total, convergence_history

In [ ]:
# Eksekusi Algoritma (10x Run), Export JSON, & Visualisasi

os.makedirs('../output_json', exist_ok=True)

semua_hasil_json = {
    "algoritma": "Ant Colony Optimization (ACO)",
    "hasil_per_klaster": {}
}

colors = {'Selatan': 'red', 'Timur': 'blue', 'Utara': 'green', 'Barat': 'purple', 'Pusat': 'orange'}
m_gabungan = folium.Map(location=[-7.275445, 112.738845], zoom_start=12, tiles='cartodbpositron')

print("🚀 MEMULAI PROSES ACO VRP (10x Run per Klaster)...\n")

for region in files_matriks_jarak.keys():
    df_jarak = pd.read_csv(files_matriks_jarak[region], index_col=0)
    df_waktu = pd.read_csv(files_matriks_waktu[region], index_col=0)

    places = df_jarak.index.tolist()
    matrix_jarak = df_jarak.values
    matrix_waktu = df_waktu.values

    # 1. Panggil fungsi skor prioritas
    prio_scores = build_priority_scores(places, coords_df)

    print("======================================================================")
    print(f"📍 MEMPROSES KLASTER {region.upper()}")
    print("-" * 70)

    # Variabel untuk melacak statistik dari 10 run
    run_fitness = []
    run_times = []
    
    global_best_dist = float('inf')
    global_best_route_idx = None
    global_best_time = float('inf')
    global_best_convergence = None
    global_best_waktu_komputasi = 0

    # Lakukan 10 kali iterasi eksekusi per klaster
    for run in range(1, 11):
        run_start_time = time.time()

        # Ubah seed setiap iterasi agar semut mengeksplorasi jalur yang berbeda
        np.random.seed(42 + run)
        random.seed(42 + run)

        # 2. Eksekusi ACO
        best_route_idx, best_dist, best_time, convergence_history = solve_aco(
            distance_matrix=matrix_jarak, 
            time_matrix=matrix_waktu, 
            priority_scores=prio_scores,
            num_ants=50,
            max_iter=150,
            alpha=1.0,
            beta=2.0,
            rho=0.1,
            Q=100
        )

        run_end_time = time.time()
        waktu_run = run_end_time - run_start_time
        
        run_fitness.append(best_dist)
        run_times.append(waktu_run)

        print(f"      ➔ Run {run}/10 Selesai | Fitness: {best_dist:.2f} | Waktu: {waktu_run:.3f} dtk")

        # Update hasil terbaik global untuk wilayah ini
        if best_dist < global_best_dist:
            global_best_dist = best_dist
            global_best_route_idx = best_route_idx
            global_best_time = best_time
            global_best_convergence = convergence_history
            global_best_waktu_komputasi = waktu_run

    # Kalkulasi statistik akhir untuk 10 run
    fitness_min = np.min(run_fitness)
    fitness_avg = np.mean(run_fitness)
    fitness_std = np.std(run_fitness, ddof=1) if len(run_fitness) > 1 else 0
    waktu_avg = np.mean(run_times)

    # 3. Proses Rute per Kurir (Hanya menggunakan rute terbaik dari 10 run)
    rute_per_kurir_list = []
    current_route_idx = []
    id_kurir = 1

    for node in global_best_route_idx:
        current_route_idx.append(node)
        if node == 0 and len(current_route_idx) > 1:
            jarak_kurir = 0; waktu_kurir = 0
            urutan = []; koords = []

            for i in range(len(current_route_idx) - 1):
                u, v = current_route_idx[i], current_route_idx[i+1]
                jarak_kurir += matrix_jarak[u][v]
                waktu_kurir += matrix_waktu[u][v]
                if v != 0: waktu_kurir += 20

            for idx_puskesmas in current_route_idx:
                nama_tempat = places[idx_puskesmas]
                urutan.append(nama_tempat)

                match = coords_df[coords_df['Nama Puskesmas'].str.strip().str.lower() == nama_tempat.strip().lower()]
                if not match.empty:
                    koords.append([match.iloc[0]['Latitude'], match.iloc[0]['Longitude']])
                else: koords.append([0.0, 0.0])

            rute_per_kurir_list.append({
                "id_kurir": id_kurir,
                "waktu_tempuh_menit": round(waktu_kurir, 2),
                "jarak_tempuh_km": round(jarak_kurir, 2),
                "urutan_kunjungan": urutan,
                "koordinat_kunjungan": koords
            })
            id_kurir += 1
            current_route_idx = [0]

    # --- PENJUMLAHAN AKURAT & OUTPUT TEKS ---
    total_waktu_akurat = sum(k["waktu_tempuh_menit"] for k in rute_per_kurir_list)
    total_jarak_akurat = sum(k["jarak_tempuh_km"] for k in rute_per_kurir_list)

    print("-" * 70)
    print(f"✅ HASIL TERBAIK KLASTER {region.upper()}:")
    print(f"Total Kurir          : {len(rute_per_kurir_list)} Orang")
    print(f"Fitness Min (Jarak)  : {round(fitness_min, 2)} KM")
    print(f"Fitness Rata-Rata    : {round(fitness_avg, 2)} KM")
    print(f"Standar Deviasi      : {round(fitness_std, 2)}")
    print(f"Waktu Komputasi (Avg): {round(waktu_avg, 3)} Detik")
    print("-" * 70)

    for kurir in rute_per_kurir_list:
        rute_singkat = ' ➔ '.join([singkat_nama(n) for n in kurir['urutan_kunjungan']])
        print(f"🚚 [KURIR {kurir['id_kurir']}] - Jarak: {kurir['jarak_tempuh_km']} KM | Waktu: {kurir['waktu_tempuh_menit']} Menit")
        print(f"   Rute: {rute_singkat}\n")

    # 4. Simpan ke JSON dengan Format Baru
    semua_hasil_json["hasil_per_klaster"][region] = {
        "statistik_10_run": {
            "fitness_minimum": round(fitness_min, 2),
            "fitness_rata_rata": round(fitness_avg, 2),
            "fitness_std_dev": round(fitness_std, 2),
            "waktu_komputasi_rata_rata_detik": round(waktu_avg, 3),
            "semua_fitness_run": [round(f, 2) for f in run_fitness]
        },
        "total_kurir": len(rute_per_kurir_list),
        "waktu_komputasi_detik_terbaik": round(global_best_waktu_komputasi, 3),
        "total_waktu_semua_menit": round(total_waktu_akurat, 2),
        "total_jarak_semua_km": round(total_jarak_akurat, 2),
        "riwayat_konvergensi": [round(v, 3) for v in global_best_convergence],
        "rute_per_kurir": rute_per_kurir_list
    }

    # 5. Plot Folium Gabungan Menggunakan Rute Terbaik
    route_names = [places[i] for i in global_best_route_idx]
    route_coords = []
    for place in route_names:
        match = coords_df[coords_df['Nama Puskesmas'].str.strip().str.lower() == place.strip().lower()]
        if not match.empty:
            lat, lon = match.iloc[0]['Latitude'], match.iloc[0]['Longitude']
            route_coords.append((lat, lon))
            folium.CircleMarker([lat, lon], radius=5, popup=place, color=colors[region], fill=True).add_to(m_gabungan)
            if 'gudang farmasi' in place.strip().lower():
                folium.Marker([lat, lon], icon=folium.Icon(color='darkblue', icon='home', prefix='fa'), popup="Gudang Farmasi").add_to(m_gabungan)

    if route_coords:
        full_road_geometry = []
        for i in range(len(route_coords) - 1):
            segment_road = get_route_geometry(route_coords[i], route_coords[i+1])
            full_road_geometry.extend(segment_road)
            time.sleep(0.05) # Jeda untuk menstabilkan API OSRM

        folium.PolyLine(full_road_geometry, color=colors[region], weight=3, opacity=0.8, tooltip=f"Rute Riil {region}").add_to(m_gabungan)

with open('../output_json/rute_aco.json', 'w') as f:
    json.dump(semua_hasil_json, f, indent=4)

print("======================================================================")
print("🎉 SEMUA KLASTER SELESAI! Data statistik 10 Run berhasil digabung ke file: ../output_json/rute_aco.json")

display(HTML("<h3>PETA GABUNGAN (SELURUH WILAYAH)</h3>"))
display(m_gabungan)